In [1]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from sklearn.model_selection import train_test_split

In [2]:
BASE_DIR = Path("clock_project")
DATA_DIR = BASE_DIR / "data"
LABELS_PATH = DATA_DIR / "labels.csv"

MODELS_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 8
EPOCHS = 30
LEARNING_RATE = 2e-4
IMAGE_SIZE = 256

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)
print("Labels exists:", LABELS_PATH.exists())

Using device: cpu
Labels exists: True


In [3]:
df = pd.read_csv(LABELS_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (100, 8)


,sample_id,digital_image_path,analog_with_hands_path,analog_clean_path,hour,minute,second,style_name
0,0,clock_project\data\digital\digital_00000.png,clock_project\data\analog_with_hands\analog_ha...,clock_project\data\analog_clean\analog_clean_0...,0,24,55,dark_mode
1,1,clock_project\data\digital\digital_00001.png,clock_project\data\analog_with_hands\analog_ha...,clock_project\data\analog_clean\analog_clean_0...,10,37,22,minimal_blue
2,2,clock_project\data\digital\digital_00002.png,clock_project\data\analog_with_hands\analog_ha...,clock_project\data\analog_clean\analog_clean_0...,17,23,3,dark_mode
3,3,clock_project\data\digital\digital_00003.png,clock_project\data\analog_with_hands\analog_ha...,clock_project\data\analog_clean\analog_clean_0...,21,35,45,dark_mode
4,4,clock_project\data\digital\digital_00004.png,clock_project\data\analog_with_hands\analog_ha...,clock_project\data\analog_clean\analog_clean_0...,14,20,46,minimal_blue


In [4]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["style_name"] if "style_name" in df.columns else None
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))

Train size: 80
Validation size: 20


In [5]:
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor()
])

In [6]:
class AnalogEraserDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        analog_img = Image.open(row["analog_with_hands_path"]).convert("RGB")
        clean_img = Image.open(row["analog_clean_path"]).convert("RGB")

        if self.transform:
            analog_img = self.transform(analog_img)
            clean_img = self.transform(clean_img)

        return analog_img, clean_img

In [7]:
train_dataset = AnalogEraserDataset(train_df, transform=transform)
val_dataset = AnalogEraserDataset(val_df, transform=transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Train batches: 10
Validation batches: 3


In [8]:
analog_batch, clean_batch = next(iter(train_loader))

print("Analog batch:", analog_batch.shape)
print("Clean batch:", clean_batch.shape)

Analog batch: torch.Size([8, 3, 256, 256])
Clean batch: torch.Size([8, 3, 256, 256])


In [11]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class ClockEraserV2(nn.Module):
    def __init__(self, base=64):
        super().__init__()

        self.enc1 = ConvBlock(3, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.enc3 = ConvBlock(base * 2, base * 4)
        self.enc4 = ConvBlock(base * 4, base * 8)

        self.pool = nn.MaxPool2d(2, 2)

        self.bottleneck = ConvBlock(base * 8, base * 8)

        self.up4 = nn.ConvTranspose2d(base * 8, base * 8, kernel_size=2, stride=2)
        self.dec4 = ConvBlock(base * 8 + base * 8, base * 4)

        self.up3 = nn.ConvTranspose2d(base * 4, base * 4, kernel_size=2, stride=2)
        self.dec3 = ConvBlock(base * 4 + base * 4, base * 2)

        self.up2 = nn.ConvTranspose2d(base * 2, base * 2, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(base * 2 + base * 2, base)

        self.up1 = nn.ConvTranspose2d(base, base, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(base + base, base)

        self.final = nn.Sequential(
            nn.Conv2d(base, 3, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        bottleneck = self.bottleneck(self.pool(e4))

        d4 = self.up4(bottleneck)
        d4 = self._match_size(d4, e4)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = self._match_size(d3, e3)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = self._match_size(d2, e2)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = self._match_size(d1, e1)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.final(d1)

    @staticmethod
    def _match_size(x, skip):
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
        return x

In [9]:
class EraserLoss(nn.Module):
    def __init__(self, hand_weight=5.0, edge_weight=2.0):
        super().__init__()

        self.hand_weight = hand_weight
        self.edge_weight = edge_weight

        sobel_x = torch.tensor(
            [[-1, 0, 1],
             [-2, 0, 2],
             [-1, 0, 1]],
            dtype=torch.float32
        ).view(1, 1, 3, 3)

        sobel_y = torch.tensor(
            [[-1, -2, -1],
             [ 0,  0,  0],
             [ 1,  2,  1]],
            dtype=torch.float32
        ).view(1, 1, 3, 3)

        self.register_buffer("sobel_x", sobel_x)
        self.register_buffer("sobel_y", sobel_y)

    def sobel_edges(self, image):
        gray = (
            0.299 * image[:, 0:1] +
            0.587 * image[:, 1:2] +
            0.114 * image[:, 2:3]
        )

        edge_x = F.conv2d(gray, self.sobel_x, padding=1)
        edge_y = F.conv2d(gray, self.sobel_y, padding=1)

        edges = torch.sqrt(edge_x ** 2 + edge_y ** 2 + 1e-6)
        return edges

    def forward(self, prediction, clean_target, analog_input):
        diff = torch.abs(analog_input - clean_target).sum(dim=1, keepdim=True)

        hand_mask = (diff > 0.05).float()

        pixel_loss = torch.abs(prediction - clean_target)

        weight_map = 1.0 + (self.hand_weight - 1.0) * hand_mask
        weighted_pixel_loss = (pixel_loss * weight_map).mean()

        prediction_edges = self.sobel_edges(prediction)
        target_edges = self.sobel_edges(clean_target)

        edge_loss = F.l1_loss(prediction_edges, target_edges)

        total_loss = weighted_pixel_loss + self.edge_weight * edge_loss

        return total_loss

In [12]:
model = ClockEraserV2(base=64).to(DEVICE)

criterion = EraserLoss(
    hand_weight=5.0,
    edge_weight=2.0
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

MODEL_PATH = MODELS_DIR / "analog_cleaner_eraser_v2.pth"
HISTORY_PATH = RESULTS_DIR / "analog_cleaner_eraser_v2_training_history.csv"

print("ClockEraserV2 ready")

ClockEraserV2 ready


In [ ]:
history = {
    "train_loss": [],
    "val_loss": []
}

best_val_loss = float("inf")

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0

    for analog_imgs, clean_imgs in train_loader:
        analog_imgs = analog_imgs.to(DEVICE)
        clean_imgs = clean_imgs.to(DEVICE)

        optimizer.zero_grad()

        predictions = model(analog_imgs)
        loss = criterion(predictions, clean_imgs, analog_imgs)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for analog_imgs, clean_imgs in val_loader:
            analog_imgs = analog_imgs.to(DEVICE)
            clean_imgs = clean_imgs.to(DEVICE)

            predictions = model(analog_imgs)
            loss = criterion(predictions, clean_imgs, analog_imgs)

            val_loss += loss.item()

    val_loss = val_loss / len(val_loader)

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.5f} | "
        f"Val Loss: {val_loss:.5f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"Saved best model | Val Loss: {best_val_loss:.5f}")

history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_PATH, index=False)

print("Training finished")
print("Best Val Loss:", best_val_loss)
print("Model saved to:", MODEL_PATH)
print("History saved to:", HISTORY_PATH)

In [ ]:
history_df = pd.DataFrame(history)

plt.figure(figsize=(8, 5))
plt.plot(history_df["train_loss"], label="Train Loss")
plt.plot(history_df["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Analog Eraser V2 — Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
best_model = ClockEraserV2(base=64).to(DEVICE)
best_model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
best_model.eval()

print("Best eraser model loaded")

In [ ]:
def tensor_to_image(tensor):
    tensor = tensor.detach().cpu().clamp(0, 1)
    return tensor.permute(1, 2, 0).numpy()


best_model.eval()

analog_imgs, clean_imgs = next(iter(val_loader))

analog_imgs = analog_imgs.to(DEVICE)
clean_imgs = clean_imgs.to(DEVICE)

with torch.no_grad():
    predictions = best_model(analog_imgs)

n = min(4, analog_imgs.size(0))

plt.figure(figsize=(4 * n, 12))

for i in range(n):
    plt.subplot(3, n, i + 1)
    plt.imshow(tensor_to_image(analog_imgs[i]))
    plt.title("Input With Hands")
    plt.axis("off")

    plt.subplot(3, n, i + 1 + n)
    plt.imshow(tensor_to_image(predictions[i]))
    plt.title("Prediction Clean")
    plt.axis("off")

    plt.subplot(3, n, i + 1 + 2 * n)
    plt.imshow(tensor_to_image(clean_imgs[i]))
    plt.title("Target Clean")
    plt.axis("off")

plt.tight_layout()
plt.show()